# Robot Aspiradora MAS con Mesa

Implementacion de los patrones basicos de sistemas multiagente usando la libreria **Mesa** (distinta a AgentPy), sobre el clasico ejemplo del robot aspiradora.


## Instalacion

Se usa `mesa` (libreria de Python para modelado basado en agentes). Se eligio porque es la libreria mas estandar del ecosistema Python para ABM, tiene buena documentacion y separa claramente el `Agent` del `Model`, lo cual mapea bien con los patrones que se piden.


In [ ]:
# Si no esta instalado, descomenta la siguiente linea:
# !pip install mesa

import random
import mesa

print('Mesa version:', mesa.__version__)


## Mini-demos (6 patrones)

### 1. Agent <-> Environment

**Libreria / mapeo:** el entorno es una clase propia (`Habitacion`, un diccionario de cuartos sucios/limpios) y el agente extiende `mesa.Agent`. En cada `step()` el agente **percibe** `entorno.sucio[self.pos]` y **actua**: limpia si esta sucio, o avanza si no.

**Prueba:** se imprime el estado del entorno antes y despues de 4 pasos, para ver como el agente lo va modificando (el entorno es 1D, una fila de cuartos, para mantenerlo simple).


In [ ]:
class Habitacion:
    """El entorno: una fila de cuartos, cada uno sucio o limpio."""
    def __init__(self, n_cuartos=5, prob_sucio=0.6, seed=1):
        rnd = random.Random(seed)
        self.sucio = {i: rnd.random() < prob_sucio for i in range(n_cuartos)}


class RobotAgent(mesa.Agent):
    """Percibe si su cuarto esta sucio y actua: limpia o se mueve."""
    def __init__(self, model, entorno, pos=0):
        super().__init__(model)
        self.entorno = entorno
        self.pos = pos

    def step(self):
        if self.entorno.sucio[self.pos]:
            self.entorno.sucio[self.pos] = False
            print(f'Robot {self.unique_id}: cuarto {self.pos} estaba sucio -> lo limpio')
        else:
            self.pos = (self.pos + 1) % len(self.entorno.sucio)
            print(f'Robot {self.unique_id}: cuarto limpio -> avanzo a {self.pos}')


class ModeloBasico(mesa.Model):
    def __init__(self, seed=1):
        super().__init__(rng=seed)
        self.entorno = Habitacion(seed=seed)
        self.robot = RobotAgent(self, self.entorno)

    def step(self):
        self.robot.step()


modelo1 = ModeloBasico()
print('Estado inicial:', modelo1.entorno.sucio)
for _ in range(4):
    modelo1.step()
print('Estado final:  ', modelo1.entorno.sucio)


Estado inicial: {0: True, 1: False, 2: False, 3: True, 4: True}
Robot 1: cuarto 0 estaba sucio -> lo limpio
Robot 1: cuarto limpio -> avanzo a 1
Robot 1: cuarto limpio -> avanzo a 2
Robot 1: cuarto limpio -> avanzo a 3
Estado final:   {0: False, 1: False, 2: False, 3: True, 4: True}


### 2. Agent with state

**Libreria / mapeo:** se agregan atributos de instancia al `mesa.Agent` (`energia`, `ultima_accion`) que persisten entre llamadas a `step()`. Mesa no obliga a nada especial para esto: el estado es simplemente memoria de instancia de Python.

**Prueba:** cuando `energia` llega a 0 el robot deja de limpiar/moverse y **descansa** en vez de actuar normalmente -> la decision cambia porque cambio el estado interno.


In [ ]:
class RobotConEstado(mesa.Agent):
    """Ademas de percibir, guarda memoria interna: energia y ultima accion."""
    def __init__(self, model, entorno, pos=0, energia=3):
        super().__init__(model)
        self.entorno = entorno
        self.pos = pos
        self.energia = energia
        self.ultima_accion = None

    def step(self):
        if self.energia <= 0:
            self.ultima_accion = 'descansar'
            self.energia += 2  # recupera energia al descansar
        elif self.entorno.sucio[self.pos]:
            self.entorno.sucio[self.pos] = False
            self.energia -= 1
            self.ultima_accion = 'limpiar'
        else:
            self.pos = (self.pos + 1) % len(self.entorno.sucio)
            self.energia -= 1
            self.ultima_accion = 'moverse'
        print(f'accion={self.ultima_accion:9s} energia={self.energia} pos={self.pos}')


class ModeloEstado(mesa.Model):
    def __init__(self, seed=2):
        super().__init__(rng=seed)
        self.entorno = Habitacion(seed=seed)
        self.robot = RobotConEstado(self, self.entorno)


modelo2 = ModeloEstado()
for _ in range(6):
    modelo2.robot.step()


accion=moverse   energia=2 pos=1
accion=moverse   energia=1 pos=2
accion=limpiar   energia=0 pos=2
accion=descansar energia=2 pos=2
accion=moverse   energia=1 pos=3
accion=limpiar   energia=0 pos=3


### 3. Multiagent system

**Libreria / mapeo:** Mesa administra automaticamente la coleccion de agentes de un modelo en `model.agents` (un `AgentSet`). `model.agents.shuffle_do('step')` activa a **todos** los robots en orden aleatorio cada paso, sin necesidad de un loop manual.

**Prueba:** se imprime la posicion de todos los robots en cada paso.


In [ ]:
class ModeloMultiagente(mesa.Model):
    def __init__(self, n_robots=4, n_cuartos=5, seed=3):
        super().__init__(rng=seed)
        self.entorno = Habitacion(n_cuartos=n_cuartos, seed=seed)
        for i in range(n_robots):
            RobotAgent(self, self.entorno, pos=i % n_cuartos)

    def step(self):
        self.agents.shuffle_do('step')


modelo3 = ModeloMultiagente()
for t in range(3):
    print(f'--- paso {t} ---')
    modelo3.step()
    posiciones = [(a.unique_id, a.pos) for a in modelo3.agents]
    print('Posiciones:', posiciones)


--- paso 0 ---
Robot 4: cuarto limpio -> avanzo a 4
Robot 1: cuarto 0 estaba sucio -> lo limpio
Robot 3: cuarto 2 estaba sucio -> lo limpio
Robot 2: cuarto 1 estaba sucio -> lo limpio
Posiciones: [(1, 0), (2, 1), (3, 2), (4, 4)]
--- paso 1 ---
Robot 1: cuarto limpio -> avanzo a 1
Robot 2: cuarto limpio -> avanzo a 2
Robot 4: cuarto limpio -> avanzo a 0
Robot 3: cuarto limpio -> avanzo a 3
Posiciones: [(1, 1), (2, 2), (3, 3), (4, 0)]
--- paso 2 ---
Robot 2: cuarto limpio -> avanzo a 3
Robot 4: cuarto limpio -> avanzo a 1
Robot 3: cuarto limpio -> avanzo a 4
Robot 1: cuarto limpio -> avanzo a 2
Posiciones: [(1, 2), (2, 3), (3, 4), (4, 1)]


### 4. Interaction between agents

**Libreria / mapeo:** cada agente consulta `self.model.agents` para ver donde estan **los demas** antes de moverse (evitar colisiones). Esto es una dependencia directa entre agentes: la posicion de uno condiciona la accion del otro.

**Prueba:** cuando el Robot 1 quiere entrar al cuarto del Robot 2, se bloquea y su contador `bloqueado` sube -> la presencia de B cambio el estado de A.


In [ ]:
class RobotConVecinos(mesa.Agent):
    """Antes de moverse revisa si otro robot ya esta en el siguiente cuarto."""
    def __init__(self, model, entorno, pos=0):
        super().__init__(model)
        self.entorno = entorno
        self.pos = pos
        self.bloqueado = 0

    def step(self):
        if self.entorno.sucio[self.pos]:
            self.entorno.sucio[self.pos] = False
            print(f'Robot {self.unique_id}: limpia cuarto {self.pos}')
            return
        siguiente = (self.pos + 1) % len(self.entorno.sucio)
        ocupado = any(o.pos == siguiente for o in self.model.agents if o is not self)
        if ocupado:
            self.bloqueado += 1
            print(f'Robot {self.unique_id}: cuarto {siguiente} ocupado -> espera (bloqueado x{self.bloqueado})')
        else:
            self.pos = siguiente
            print(f'Robot {self.unique_id}: avanza a {siguiente}')


class ModeloVecinos(mesa.Model):
    def __init__(self, seed=5):
        super().__init__(rng=seed)
        # todo limpio para forzar que ambos robots intenten moverse
        self.entorno = Habitacion(n_cuartos=3, prob_sucio=0.0, seed=seed)
        self.r1 = RobotConVecinos(self, self.entorno, pos=0)
        self.r2 = RobotConVecinos(self, self.entorno, pos=1)

    def step(self):
        self.agents.shuffle_do('step')


modelo4 = ModeloVecinos()
for t in range(3):
    print(f'--- paso {t} ---')
    modelo4.step()


--- paso 0 ---
Robot 1: cuarto 1 ocupado -> espera (bloqueado x1)
Robot 2: avanza a 2
--- paso 1 ---
Robot 1: avanza a 1
Robot 2: avanza a 0
--- paso 2 ---
Robot 2: cuarto 1 ocupado -> espera (bloqueado x1)
Robot 1: avanza a 2


### 5. Reactive agent

**Libreria / mapeo:** el `step()` del agente **no usa ninguna memoria** (no hay atributos de estado mas alla de posicion/direccion): la decision sale directo de la percepcion del frame actual, siguiendo la regla `si obstaculo -> gira; si libre -> avanza`.

**Prueba:** reglas sencillas sobre un pasillo con un obstaculo fijo.


In [ ]:
class RobotReactivo(mesa.Agent):
    """Puramente reactivo: percepcion actual -> regla -> accion. Sin planeacion."""
    def __init__(self, model, pasillo, pos=0, direccion=1):
        super().__init__(model)
        self.pasillo = pasillo  # lista de bool, True = obstaculo
        self.pos = pos
        self.direccion = direccion

    def step(self):
        siguiente = self.pos + self.direccion
        hay_obstaculo = siguiente < 0 or siguiente >= len(self.pasillo) or self.pasillo[siguiente]
        if hay_obstaculo:
            self.direccion *= -1
            print(f'Obstaculo adelante -> giro, direccion={self.direccion}')
        else:
            self.pos = siguiente
            print(f'Libre adelante -> avanzo a {self.pos}')


class ModeloReactivo(mesa.Model):
    def __init__(self):
        super().__init__()
        pasillo = [False, False, True, False, False]  # obstaculo en indice 2
        self.robot = RobotReactivo(self, pasillo, pos=0)


modelo5 = ModeloReactivo()
for _ in range(5):
    modelo5.robot.step()


Libre adelante -> avanzo a 1
Obstaculo adelante -> giro, direccion=-1
Libre adelante -> avanzo a 0
Obstaculo adelante -> giro, direccion=1
Libre adelante -> avanzo a 1


### 6. Communication & negotiation

**Libreria / mapeo:** no se usa un modulo de mensajeria de Mesa (no trae uno nativo); se implementa un intercambio minimo de mensajes en Python puro entre agentes: cada robot 'ofrece' su distancia al cuarto sucio y gana el mas cercano (negociacion tipo subasta).

**Prueba:** log de ofertas y el acuerdo final.


In [ ]:
class RobotNegociador(mesa.Agent):
    def __init__(self, model, pos):
        super().__init__(model)
        self.pos = pos

    def distancia(self, objetivo):
        return abs(self.pos - objetivo)


def negociar_limpieza(robots, cuarto_sucio):
    """Cada robot 'grita' su distancia al cuarto sucio; gana el mas cercano."""
    ofertas = {}
    for r in robots:
        d = r.distancia(cuarto_sucio)
        ofertas[r.unique_id] = d
        print(f'Robot {r.unique_id} ofrece: distancia={d}')
    ganador_id = min(ofertas, key=ofertas.get)
    print(f'Acuerdo: Robot {ganador_id} limpia el cuarto {cuarto_sucio} (menor distancia)')
    return ganador_id


class ModeloNegociacion(mesa.Model):
    def __init__(self):
        super().__init__()
        self.robots = [RobotNegociador(self, pos=p) for p in [0, 4, 2]]


modelo6 = ModeloNegociacion()
_ = negociar_limpieza(modelo6.robots, cuarto_sucio=3)


Robot 1 ofrece: distancia=3
Robot 2 ofrece: distancia=1
Robot 3 ofrece: distancia=1
Acuerdo: Robot 2 limpia el cuarto 3 (menor distancia)


## Conclusiones
Con Mesa se pudieron implementar los 6 patrones basicos de un sistema multiagente reutilizando el mismo entorno (`Habitacion`) con las mismas clase falta, la conclucion (`mesa.Agent`),  las variables solo la logica que es `step()` y cuando hizo falta, consultando `model.agents` para cada uno de los robor que pueda ver por decirlo de un modo a los demas con eso el notebook corre en un par de segundos.


### Comparativa corta: Mesa vs AgentPy

Para que no llegue a programar nada en el AgentPy, asi que esto es mas una se usa la comparacion de diseno que solo leyendo la documentacion de ambas que una comparacion para poder "sentir" el codigo que es Mesa obliga a un estilo mucho mas orientado a objeto que hay que declarar clases tales como`Agent` y `Model` explicitas y heredar con eso cosas de ellas, lo cual es un poco mas se puede decir verboso al inicio pero deja con esto el codigo ordenado y facil de extender (agregar mas robots fue una linea). AgentPy, con eso lo que vi en su documentacion, esta pensado poco a poco desde el diseno para correr experimentos y diferentes barridos de parametros (`ap.Experiment`, `ap.Sample`), con esto asi que ahi abstrae mas que Mesa, pero con el cambio exige aprender su propia estructura de parametros y con esto que se pueda ver los registro de variables. Para un demo chico tal y como este lo que mas costo de esto es Mesa fue el arranque, pero a favor podemos ver quien tiene cada estado de cada agente queda mejor explicito y facil de poder imprimir para depurar que es justo lo que necesitamos para estas pruebas.
